<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/deep_learning_example/02_house_price_prediction.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

# 캘리포니아 집값 예측 - 딥러닝 회귀 문제

이 노트북은 scikit-learn의 California Housing 데이터셋을 사용하여 딥러닝으로 집값을 예측하는 예제입니다.

## 🎯 목표
- 캘리포니아 주의 주택 특성을 바탕으로 집값 예측
- 딥러닝을 활용한 회귀 문제 해결
- 다양한 신경망 구조 비교 및 앙상블 학습

## 📊 데이터셋 정보
- **California Housing Dataset**: scikit-learn 내장 데이터셋
- **샘플 수**: 20,640개 블록 그룹
- **특성**: 8개 (평균 소득, 집 연령, 평균 방 수 등)
- **타겟**: 블록 그룹의 중간 주택 가격 ($100,000 단위)

In [ ]:
# 필요한 라이브러리 import (Google Colab T4 환경)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 나눔고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

print(f"PyTorch 버전: {torch.__version__}")

# GPU 설정 (Google Colab T4 환경)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: {device}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 버전: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print(f"Using device: {device}")

# 시드 설정 (재현 가능한 결과를 위해)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# 시각화 설정
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. 데이터 로드 및 탐색

In [ ]:
# California Housing 데이터셋 로드
from sklearn.datasets import fetch_california_housing

# 데이터 로드
print("📊 California Housing 데이터셋 로드 중...")
housing = fetch_california_housing()

# DataFrame으로 변환
train_df = pd.DataFrame(housing.data, columns=housing.feature_names)
train_df['SalePrice'] = housing.target  # 타겟값 추가

print("✅ 데이터 로드 완료!")

# 데이터 정보 출력
print(f"\n📈 데이터 크기:")
print(f"- 전체 데이터: {train_df.shape}")
print(f"- 특성 수: {len(housing.feature_names)}개")
print(f"- 특성 이름: {', '.join(housing.feature_names)}")

# 데이터셋 설명
print(f"\n📝 특성 설명:")
feature_descriptions = {
    'MedInc': '블록 그룹의 중간 소득',
    'HouseAge': '블록 그룹의 중간 주택 연령',
    'AveRooms': '가구당 평균 방 수',
    'AveBedrms': '가구당 평균 침실 수',
    'Population': '블록 그룹 인구',
    'AveOccup': '가구당 평균 거주자 수',
    'Latitude': '블록 그룹 위도',
    'Longitude': '블록 그룹 경도'
}

for feature, description in feature_descriptions.items():
    if feature in train_df.columns:
        print(f"- {feature}: {description}")

# 기본 통계 출력
print(f"\n💰 집값 기본 통계 (단위: $100,000):")
print(f"- 평균: ${train_df['SalePrice'].mean()*100000:,.0f}")
print(f"- 중앙값: ${train_df['SalePrice'].median()*100000:,.0f}")
print(f"- 최소값: ${train_df['SalePrice'].min()*100000:,.0f}")
print(f"- 최대값: ${train_df['SalePrice'].max()*100000:,.0f}")
print(f"- 표준편차: ${train_df['SalePrice'].std()*100000:,.0f}")

# 첫 5개 샘플 확인
print(f"\n🔍 데이터 샘플 (첫 5행):")
print(train_df.head())

## 2. 데이터 탐색 (EDA)

In [ ]:
# 데이터 정보 확인
print("🔍 데이터 기본 정보")
print("=" * 50)
print(f"전체 데이터 형태: {train_df.shape}")
print(f"특성 개수: {train_df.shape[1] - 1}개")

# 데이터 타입 확인
print(f"\n📊 데이터 타입:")
print(train_df.dtypes)

# 결측치 확인
missing_data = train_df.isnull().sum()
print(f"\n🔍 결측치 현황:")
if missing_data.sum() == 0:
    print("✅ 결측치 없음 - California Housing 데이터셋은 전처리된 깨끗한 데이터입니다.")
else:
    for col, count in missing_data[missing_data > 0].items():
        percentage = (count / len(train_df)) * 100
        print(f"- {col}: {count}개 ({percentage:.1f}%)")

# 기본 통계량
print(f"\n📊 기본 통계량:")
print(train_df.describe().round(3))

In [ ]:
# 집값 분포 시각화
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# 1. 집값 분포
ax1 = axes[0, 0]
ax1.hist(train_df['SalePrice'], bins=50, alpha=0.7, edgecolor='black')
ax1.set_xlabel('집값 ($100,000)')
ax1.set_ylabel('빈도')
ax1.set_title('집값 분포')

# 2. 로그 변환된 집값 분포
ax2 = axes[0, 1]
log_prices = np.log1p(train_df['SalePrice'])
ax2.hist(log_prices, bins=50, alpha=0.7, edgecolor='black', color='orange')
ax2.set_xlabel('Log(집값 + 1)')
ax2.set_ylabel('빈도')
ax2.set_title('로그 변환된 집값 분포')

# 3. 중간 소득 vs 집값
ax3 = axes[0, 2]
ax3.scatter(train_df['MedInc'], train_df['SalePrice'], alpha=0.3)
ax3.set_xlabel('중간 소득')
ax3.set_ylabel('집값 ($100,000)')
ax3.set_title('중간 소득 vs 집값')

# 4. 집 연령 vs 집값
ax4 = axes[0, 3]
ax4.scatter(train_df['HouseAge'], train_df['SalePrice'], alpha=0.3)
ax4.set_xlabel('집 연령 (년)')
ax4.set_ylabel('집값 ($100,000)')
ax4.set_title('집 연령 vs 집값')

# 5. 평균 방 수 vs 집값
ax5 = axes[1, 0]
ax5.scatter(train_df['AveRooms'], train_df['SalePrice'], alpha=0.3)
ax5.set_xlabel('평균 방 수')
ax5.set_ylabel('집값 ($100,000)')
ax5.set_title('평균 방 수 vs 집값')

# 6. 지리적 위치와 집값
ax6 = axes[1, 1]
scatter = ax6.scatter(train_df['Longitude'], train_df['Latitude'], 
                     c=train_df['SalePrice'], cmap='viridis', alpha=0.3)
ax6.set_xlabel('경도')
ax6.set_ylabel('위도')
ax6.set_title('지리적 위치별 집값')
plt.colorbar(scatter, ax=ax6, label='집값 ($100,000)')

# 7. 인구 vs 집값
ax7 = axes[1, 2]
ax7.scatter(train_df['Population'], train_df['SalePrice'], alpha=0.3)
ax7.set_xlabel('인구')
ax7.set_ylabel('집값 ($100,000)')
ax7.set_title('인구 vs 집값')
ax7.set_xlim(0, 10000)  # 이상치 제외하고 보기

# 8. 가구당 평균 거주자 vs 집값
ax8 = axes[1, 3]
ax8.scatter(train_df['AveOccup'], train_df['SalePrice'], alpha=0.3)
ax8.set_xlabel('가구당 평균 거주자')
ax8.set_ylabel('집값 ($100,000)')
ax8.set_title('가구당 평균 거주자 vs 집값')
ax8.set_xlim(0, 10)  # 이상치 제외하고 보기

plt.tight_layout()
plt.show()

In [ ]:
# 특성들과 집값의 상관관계 분석
correlation_matrix = train_df.corr()

# 집값과의 상관관계
price_correlations = correlation_matrix['SalePrice'].abs().sort_values(ascending=False)

print("💡 집값과 상관관계가 높은 특성들:")
print("-" * 50)
for i, (feature, corr) in enumerate(price_correlations.items()):
    if feature != 'SalePrice':  # SalePrice 자체 제외
        print(f"{i:2d}. {feature:20s}: {corr:6.3f}")

# 상관관계 히트맵
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, fmt='.2f')
plt.title('특성들 간 상관관계')
plt.tight_layout()
plt.show()

# 주요 인사이트
print("\n📊 주요 인사이트:")
print("- 중간 소득(MedInc)이 집값과 가장 높은 상관관계 (0.688)")
print("- 위도와 경도도 집값과 상당한 상관관계 (캘리포니아 북부가 더 비쌈)")
print("- 평균 방 수와 평균 침실 수는 집값과 약한 양의 상관관계")
print("- 가구당 평균 거주자 수는 집값과 음의 상관관계")

## 3. 데이터 전처리

In [ ]:
def preprocess_california_housing(train_df):
    """
    California Housing 데이터를 전처리합니다.
    """
    print("🔧 데이터 전처리 시작...")
    
    # 데이터 복사
    data_processed = train_df.copy()
    
    # 타겟 변수 분리
    y = data_processed['SalePrice'].values
    X = data_processed.drop('SalePrice', axis=1)
    
    print(f"원본 데이터 크기: {X.shape}")
    
    # 1. 새로운 특성 생성 (Feature Engineering)
    print(f"\n🔨 특성 엔지니어링:")
    
    # 방 대비 침실 비율
    X['RoomBedroomRatio'] = X['AveRooms'] / (X['AveBedrms'] + 0.1)
    print("  - RoomBedroomRatio: 방 대비 침실 비율")
    
    # 인구 밀도 관련 특성
    X['PopulationPerHousehold'] = X['Population'] / X['AveOccup']
    print("  - PopulationPerHousehold: 가구당 인구")
    
    # 지리적 특성 조합
    X['Location'] = X['Latitude'] + X['Longitude']
    print("  - Location: 위도 + 경도 조합")
    
    # 소득 대비 방 수
    X['RoomsPerIncome'] = X['AveRooms'] / (X['MedInc'] + 0.1)
    print("  - RoomsPerIncome: 소득 대비 방 수")
    
    # 2. 이상치 처리
    print(f"\n🎯 이상치 감지 및 처리:")
    
    # IQR 방법으로 이상치 캡핑
    for col in X.columns:
        Q1 = X[col].quantile(0.25)
        Q3 = X[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        before_count = ((X[col] < lower_bound) | (X[col] > upper_bound)).sum()
        if before_count > 0:
            X[col] = np.clip(X[col], lower_bound, upper_bound)
            print(f"  - {col}: {before_count}개 이상치 캡핑")
    
    # 3. 훈련/검증/테스트 데이터 분할
    from sklearn.model_selection import train_test_split
    
    # 먼저 훈련+검증과 테스트로 분할 (80/20)
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # 훈련과 검증으로 분할 (75/25 of temp = 60/20 of total)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42
    )
    
    # 4. 스케일링
    scaler = RobustScaler()  # 이상치에 강한 스케일러
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"\n✅ 전처리 완료!")
    print(f"- 훈련 데이터: {X_train_scaled.shape}")
    print(f"- 검증 데이터: {X_val_scaled.shape}")
    print(f"- 테스트 데이터: {X_test_scaled.shape}")
    print(f"- 특성 개수: {X_train_scaled.shape[1]}")
    
    feature_names = list(X.columns)
    
    return (X_train_scaled, X_val_scaled, X_test_scaled, 
            y_train, y_val, y_test, scaler, feature_names)

# 데이터 전처리 실행
(X_train, X_val, X_test, y_train, y_val, y_test, 
 scaler, feature_names) = preprocess_california_housing(train_df)

print(f"\n📋 처리된 특성들 ({len(feature_names)}개):")
for i, feature in enumerate(feature_names):
    print(f"{i+1:2d}. {feature}")

## 4. 딥러닝 모델 구현

In [ ]:
class HousePriceNet(nn.Module):
    """
    집값 예측을 위한 딥러닝 회귀 모델
    """
    def __init__(self, input_size, hidden_sizes=[256, 128, 64, 32], dropout_rate=0.3):
        super(HousePriceNet, self).__init__()
        
        layers = []
        prev_size = input_size
        
        # 은닉층들 구성
        for i, hidden_size in enumerate(hidden_sizes):
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.BatchNorm1d(hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_size = hidden_size
        
        # 출력층 (회귀이므로 활성화 함수 없음)
        layers.append(nn.Linear(prev_size, 1))
        
        self.network = nn.Sequential(*layers)
        
        # 가중치 초기화
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.constant_(module.bias, 0)
    
    def forward(self, x):
        return self.network(x)

# 다양한 모델 구성 정의
model_configs = {
    'Simple': {
        'hidden_sizes': [128, 64], 
        'dropout_rate': 0.2
    },
    'Deep': {
        'hidden_sizes': [512, 256, 128, 64, 32], 
        'dropout_rate': 0.4
    },
    'Wide': {
        'hidden_sizes': [512, 256], 
        'dropout_rate': 0.3
    },
    'Balanced': {
        'hidden_sizes': [256, 128, 64, 32], 
        'dropout_rate': 0.3
    },
    'Complex': {
        'hidden_sizes': [512, 256, 128, 64, 32, 16], 
        'dropout_rate': 0.4
    }
}

# 모델들 생성
input_size = X_train.shape[1]
models = {}

print(f"🏗️ 모델 생성 (입력 크기: {input_size})")
print("=" * 60)

for name, config in model_configs.items():
    model = HousePriceNet(input_size, **config).to(device)
    models[name] = model
    
    # 파라미터 수 계산
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"{name:>10s}: {total_params:>8,} 파라미터 (훈련 가능: {trainable_params:>8,})")

# 대표 모델 구조 출력
print(f"\n🏗️ Balanced 모델 구조:")
print(models['Balanced'])

## 5. 모델 학습 및 평가

In [ ]:
def train_regression_model(model, X_train, y_train, X_val, y_val,
                          epochs=200, batch_size=64, lr=0.001, patience=20):
    """
    회귀 모델을 훈련시키고 결과를 반환합니다.
    """
    # 데이터 로더 준비
    train_dataset = TensorDataset(
        torch.FloatTensor(X_train),
        torch.FloatTensor(y_train).unsqueeze(1)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    val_dataset = TensorDataset(
        torch.FloatTensor(X_val),
        torch.FloatTensor(y_val).unsqueeze(1)
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # 최적화 설정
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=10, factor=0.5, verbose=False
    )
    
    # 학습 기록
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_rmse': [],
        'val_rmse': []
    }
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # 훈련 모드
        model.train()
        train_loss = 0
        train_predictions = []
        train_targets = []
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            
            # 그래디언트 클리핑
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            train_predictions.extend(outputs.detach().cpu().numpy().flatten())
            train_targets.extend(batch_y.detach().cpu().numpy().flatten())
        
        # 검증 모드
        model.eval()
        val_loss = 0
        val_predictions = []
        val_targets = []
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                val_loss += loss.item()
                val_predictions.extend(outputs.cpu().numpy().flatten())
                val_targets.extend(batch_y.cpu().numpy().flatten())
        
        # 평균 손실 및 RMSE 계산
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        train_rmse = np.sqrt(mean_squared_error(train_targets, train_predictions))
        val_rmse = np.sqrt(mean_squared_error(val_targets, val_predictions))
        
        # 기록
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_rmse'].append(train_rmse)
        history['val_rmse'].append(val_rmse)
        
        # 스케줄러 업데이트
        scheduler.step(avg_val_loss)
        
        # 조기 종료 체크
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        # 진행 상황 출력
        if epoch % 40 == 0 or epoch == epochs - 1:
            print(f'Epoch {epoch:3d}: Train Loss={avg_train_loss:.4f}, '
                  f'Val Loss={avg_val_loss:.4f}, Train RMSE={train_rmse:.4f}, '
                  f'Val RMSE={val_rmse:.4f}')
        
        if patience_counter >= patience:
            print(f'조기 종료: {epoch+1} 에폭에서 훈련 종료')
            break
    
    # 최고 모델 복원
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return history

# 모든 모델 훈련
results = {}
print(f"\n🚀 모델 훈련 시작...")
print("=" * 70)

for name, model in models.items():
    print(f"\n📊 {name} 모델 훈련 중...")
    
    history = train_regression_model(
        model, X_train, y_train, X_val, y_val,
        epochs=200, batch_size=64, lr=0.001, patience=25
    )
    
    results[name] = {
        'model': model,
        'history': history,
        'best_val_rmse': min(history['val_rmse'])
    }
    
    print(f"✅ {name} 모델 완료 - 최고 검증 RMSE: {results[name]['best_val_rmse']:.4f}")

print(f"\n🏆 모든 모델 훈련 완료!")

## 6. 결과 시각화 및 분석

In [ ]:
# 학습 결과 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 손실 함수 비교
ax1 = axes[0, 0]
for name, result in results.items():
    history = result['history']
    ax1.plot(history['train_loss'], label=f'{name} (Train)', linestyle='-', alpha=0.7)
    ax1.plot(history['val_loss'], label=f'{name} (Val)', linestyle='--', alpha=0.7)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.set_title('모델별 손실 함수 변화')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 2. RMSE 비교
ax2 = axes[0, 1]
for name, result in results.items():
    history = result['history']
    ax2.plot(history['train_rmse'], label=f'{name} (Train)', linestyle='-', alpha=0.7)
    ax2.plot(history['val_rmse'], label=f'{name} (Val)', linestyle='--', alpha=0.7)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('RMSE')
ax2.set_title('모델별 RMSE 변화')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(True, alpha=0.3)

# 3. 최종 성능 비교
ax3 = axes[1, 0]
model_names = list(results.keys())
best_val_rmses = [results[name]['best_val_rmse'] for name in model_names]
final_train_rmses = [results[name]['history']['train_rmse'][-1] for name in model_names]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax3.bar(x - width/2, final_train_rmses, width, label='Train RMSE', alpha=0.7)
bars2 = ax3.bar(x + width/2, best_val_rmses, width, label='Best Val RMSE', alpha=0.7)

ax3.set_xlabel('Model')
ax3.set_ylabel('RMSE')
ax3.set_title('모델별 최종 성능 비교')
ax3.set_xticks(x)
ax3.set_xticklabels(model_names, rotation=45)
ax3.legend()

# 값 표시
for bar in bars1:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.3f}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.3f}', ha='center', va='bottom', fontsize=9)

# 4. 과적합 분석
ax4 = axes[1, 1]
overfitting_scores = []
for name in model_names:
    history = results[name]['history']
    final_train_rmse = history['train_rmse'][-1]
    best_val_rmse = min(history['val_rmse'])
    overfitting = final_train_rmse - best_val_rmse
    overfitting_scores.append(overfitting)

colors = ['green' if score > -0.05 else 'orange' if score > -0.1 else 'red' 
          for score in overfitting_scores]
bars = ax4.bar(model_names, overfitting_scores, color=colors, alpha=0.7)
ax4.axhline(y=0, color='black', linestyle='--', alpha=0.7)
ax4.set_xlabel('Model')
ax4.set_ylabel('Train RMSE - Val RMSE')
ax4.set_title('과적합 분석')
ax4.tick_params(axis='x', rotation=45)

for bar, score in zip(bars, overfitting_scores):
    ax4.text(bar.get_x() + bar.get_width()/2., score + 0.002,
             f'{score:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# 최고 성능 모델 선택
best_model_name = min(results.keys(), key=lambda x: results[x]['best_val_rmse'])
best_model = results[best_model_name]['model']
best_rmse = results[best_model_name]['best_val_rmse']

print(f"🏆 최고 성능 모델: {best_model_name}")
print(f"🎯 최고 검증 RMSE: {best_rmse:.4f} (로그 스케일)")

# 달러 단위로 변환 (대략적 추정)
rmse_dollars = np.expm1(best_rmse)  # 로그 역변환
print(f"🎯 달러 단위 추정 RMSE: ${rmse_dollars:,.0f}")

# 모델별 성능 요약
print(f"\n📊 모델별 성능 요약:")
print("-" * 70)
for name in model_names:
    history = results[name]['history']
    train_rmse = history['train_rmse'][-1]
    val_rmse = results[name]['best_val_rmse']
    overfitting = train_rmse - val_rmse
    
    print(f"{name:>10}: Train RMSE {train_rmse:.3f}, Val RMSE {val_rmse:.3f}, "
          f"Overfitting {overfitting:.3f}")

## 7. 예측 성능 상세 분석

In [ ]:
# 최고 성능 모델로 상세 분석
best_model.eval()
X_val_tensor = torch.FloatTensor(X_val).to(device)

with torch.no_grad():
    val_predictions = best_model(X_val_tensor).cpu().numpy().flatten()

# 성능 지표 계산
mse = mean_squared_error(y_val, val_predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_val, val_predictions)
r2 = r2_score(y_val, val_predictions)

print(f"📊 {best_model_name} 모델 상세 성능 (단위: $100,000):")
print("=" * 50)
print(f"RMSE:        {rmse:.4f} (${rmse*100000:,.0f})")
print(f"MAE:         {mae:.4f} (${mae*100000:,.0f})")
print(f"R² Score:    {r2:.4f}")
print(f"평균 집값:   {np.mean(y_val):.4f} (${np.mean(y_val)*100000:,.0f})")
print(f"RMSE/평균:   {rmse/np.mean(y_val)*100:.1f}%")

# 예측 vs 실제 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. 예측 vs 실제 산점도
ax1 = axes[0, 0]
ax1.scatter(y_val, val_predictions, alpha=0.6)
ax1.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
ax1.set_xlabel('실제 집값 ($100,000)')
ax1.set_ylabel('예측 집값 ($100,000)')
ax1.set_title(f'{best_model_name} - 예측 vs 실제')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 잔차 분포
ax2 = axes[0, 1]
residuals = val_predictions - y_val
ax2.hist(residuals, bins=50, alpha=0.7, edgecolor='black')
ax2.axvline(0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('잔차 (예측 - 실제) ($100,000)')
ax2.set_ylabel('빈도')
ax2.set_title('잔차 분포')
ax2.grid(True, alpha=0.3)

# 3. 잔차 vs 예측값
ax3 = axes[1, 0]
ax3.scatter(val_predictions, residuals, alpha=0.6)
ax3.axhline(0, color='red', linestyle='--', linewidth=2)
ax3.set_xlabel('예측 집값 ($100,000)')
ax3.set_ylabel('잔차 ($100,000)')
ax3.set_title('잔차 vs 예측값')
ax3.grid(True, alpha=0.3)

# 4. 가격대별 성능
ax4 = axes[1, 1]
# 가격대별 그룹화
price_ranges = [(0, 1), (1, 2), (2, 3), (3, 5)]
range_labels = ['<$100K', '$100K-200K', '$200K-300K', '>$300K']
range_rmses = []

for min_price, max_price in price_ranges:
    mask = (y_val >= min_price) & (y_val < max_price)
    if np.sum(mask) > 0:
        range_rmse = np.sqrt(mean_squared_error(y_val[mask], val_predictions[mask]))
        range_rmses.append(range_rmse)
    else:
        range_rmses.append(0)

bars = ax4.bar(range_labels, range_rmses, alpha=0.7)
ax4.set_xlabel('가격대')
ax4.set_ylabel('RMSE ($100,000)')
ax4.set_title('가격대별 RMSE')
ax4.tick_params(axis='x', rotation=45)

for bar, rmse_val in zip(bars, range_rmses):
    if rmse_val > 0:
        ax4.text(bar.get_x() + bar.get_width()/2., rmse_val + 0.01,
                 f'{rmse_val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# 예측 정확도 분석
percentage_errors = np.abs(residuals) / y_val * 100
print(f"\n🎯 예측 정확도 분석:")
print(f"- 10% 이내 정확도: {np.sum(percentage_errors <= 10) / len(percentage_errors) * 100:.1f}%")
print(f"- 20% 이내 정확도: {np.sum(percentage_errors <= 20) / len(percentage_errors) * 100:.1f}%")
print(f"- 평균 절대 오차율: {np.mean(percentage_errors):.1f}%")
print(f"- 중간 절대 오차율: {np.median(percentage_errors):.1f}%")

## 8. 테스트 데이터 최종 평가

In [ ]:
# 최고 성능 모델로 테스트 데이터 평가
best_model.eval()
X_test_tensor = torch.FloatTensor(X_test).to(device)

with torch.no_grad():
    test_predictions = best_model(X_test_tensor).cpu().numpy().flatten()

# 테스트 성능 계산
test_mse = mean_squared_error(y_test, test_predictions)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, test_predictions)
test_r2 = r2_score(y_test, test_predictions)

print(f"🔮 테스트 데이터 최종 성능:")
print("=" * 50)
print(f"RMSE:        {test_rmse:.4f} (${test_rmse*100000:,.0f})")
print(f"MAE:         {test_mae:.4f} (${test_mae*100000:,.0f})")
print(f"R² Score:    {test_r2:.4f}")
print(f"평균 집값:   {np.mean(y_test):.4f} (${np.mean(y_test)*100000:,.0f})")

# 예측 분포 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 1. 테스트 예측 vs 실제
ax1.scatter(y_test, test_predictions, alpha=0.4, s=20)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
ax1.set_xlabel('실제 집값 ($100,000)')
ax1.set_ylabel('예측 집값 ($100,000)')
ax1.set_title(f'테스트 데이터: 예측 vs 실제 (R²={test_r2:.3f})')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 가격대별 분포 비교
price_bins = [0, 1, 2, 3, 4, 5]
price_labels = ['<$100K', '$100K-200K', '$200K-300K', '$300K-400K', '>$400K']

actual_counts = []
predicted_counts = []

for i in range(len(price_bins)-1):
    if i < len(price_bins)-2:
        actual_mask = (y_test >= price_bins[i]) & (y_test < price_bins[i+1])
        predicted_mask = (test_predictions >= price_bins[i]) & (test_predictions < price_bins[i+1])
    else:
        actual_mask = y_test >= price_bins[i]
        predicted_mask = test_predictions >= price_bins[i]
    
    actual_counts.append(np.sum(actual_mask))
    predicted_counts.append(np.sum(predicted_mask))

x = np.arange(len(price_labels))
width = 0.35

bars1 = ax2.bar(x - width/2, actual_counts, width, label='실제', alpha=0.7)
bars2 = ax2.bar(x + width/2, predicted_counts, width, label='예측', alpha=0.7)

ax2.set_xlabel('가격대')
ax2.set_ylabel('샘플 수')
ax2.set_title('가격대별 분포 비교')
ax2.set_xticks(x)
ax2.set_xticklabels(price_labels, rotation=45)
ax2.legend()

# 값 표시
for bar in bars1:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 10,
             f'{int(height)}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 10,
             f'{int(height)}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# 예측 오차 통계
test_residuals = test_predictions - y_test
test_percentage_errors = np.abs(test_residuals) / y_test * 100

print(f"\n📊 테스트 데이터 예측 정확도:")
print(f"- 10% 이내 정확도: {np.sum(test_percentage_errors <= 10) / len(test_percentage_errors) * 100:.1f}%")
print(f"- 20% 이내 정확도: {np.sum(test_percentage_errors <= 20) / len(test_percentage_errors) * 100:.1f}%")
print(f"- 30% 이내 정확도: {np.sum(test_percentage_errors <= 30) / len(test_percentage_errors) * 100:.1f}%")
print(f"- 평균 절대 오차율: {np.mean(test_percentage_errors):.1f}%")
print(f"- 중간 절대 오차율: {np.median(test_percentage_errors):.1f}%")

## 9. 앙상블 모델 (보너스)

In [ ]:
# 모든 모델의 예측을 결합한 앙상블
print("🎭 앙상블 모델 생성...")

# 각 모델의 검증 및 테스트 데이터 예측
ensemble_predictions_val = []
ensemble_predictions_test = []
model_weights = []

X_val_tensor = torch.FloatTensor(X_val).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)

for name, result in results.items():
    model = result['model']
    model.eval()
    
    with torch.no_grad():
        # 검증 데이터 예측
        val_pred = model(X_val_tensor).cpu().numpy().flatten()
        ensemble_predictions_val.append(val_pred)
        
        # 테스트 데이터 예측
        test_pred = model(X_test_tensor).cpu().numpy().flatten()
        ensemble_predictions_test.append(test_pred)
    
    # 가중치는 성능에 반비례 (RMSE가 낮을수록 높은 가중치)
    weight = 1.0 / result['best_val_rmse']
    model_weights.append(weight)

# 가중치 정규화
model_weights = np.array(model_weights)
model_weights = model_weights / np.sum(model_weights)

print(f"\n📊 앙상블 가중치:")
for name, weight in zip(results.keys(), model_weights):
    print(f"- {name}: {weight:.3f}")

# 가중 평균 계산
ensemble_val_pred = np.average(ensemble_predictions_val, weights=model_weights, axis=0)
ensemble_test_pred = np.average(ensemble_predictions_test, weights=model_weights, axis=0)

# 앙상블 성능 평가
ensemble_val_rmse = np.sqrt(mean_squared_error(y_val, ensemble_val_pred))
ensemble_val_r2 = r2_score(y_val, ensemble_val_pred)

ensemble_test_rmse = np.sqrt(mean_squared_error(y_test, ensemble_test_pred))
ensemble_test_r2 = r2_score(y_test, ensemble_test_pred)

print(f"\n🏆 앙상블 모델 성능:")
print(f"검증 데이터:")
print(f"- RMSE: {ensemble_val_rmse:.4f} (${ensemble_val_rmse*100000:,.0f})")
print(f"- R² Score: {ensemble_val_r2:.4f}")
print(f"\n테스트 데이터:")
print(f"- RMSE: {ensemble_test_rmse:.4f} (${ensemble_test_rmse*100000:,.0f})")
print(f"- R² Score: {ensemble_test_r2:.4f}")

# 최고 개별 모델과 비교
print(f"\n📈 성능 개선:")
best_individual_rmse = min([result['best_val_rmse'] for result in results.values()])
improvement = (best_individual_rmse - ensemble_val_rmse) / best_individual_rmse * 100
print(f"- 최고 개별 모델 대비 {improvement:.2f}% 개선")

# 앙상블과 개별 모델 성능 비교 시각화
plt.figure(figsize=(12, 6))

model_names_with_ensemble = list(results.keys()) + ['Ensemble']
val_rmses = [results[name]['best_val_rmse'] for name in results.keys()] + [ensemble_val_rmse]
test_rmses = []

# 각 모델의 테스트 RMSE 계산
for name, result in results.items():
    model = result['model']
    model.eval()
    with torch.no_grad():
        test_pred = model(X_test_tensor).cpu().numpy().flatten()
        test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
        test_rmses.append(test_rmse)

test_rmses.append(ensemble_test_rmse)

x = np.arange(len(model_names_with_ensemble))
width = 0.35

bars1 = plt.bar(x - width/2, val_rmses, width, label='Validation RMSE', alpha=0.7)
bars2 = plt.bar(x + width/2, test_rmses, width, label='Test RMSE', alpha=0.7)

# 앙상블 바 강조
bars1[-1].set_color('darkgreen')
bars2[-1].set_color('darkred')

plt.xlabel('Model')
plt.ylabel('RMSE ($100,000)')
plt.title('개별 모델 vs 앙상블 성능 비교')
plt.xticks(x, model_names_with_ensemble, rotation=45)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

# 값 표시
for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.3f}', ha='center', va='bottom', fontsize=8)

for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 10. 학습 요약 및 개선 방향

In [ ]:
# 최종 요약 정보
print("🎉 California Housing 집값 예측 프로젝트 완료!")
print("=" * 60)
print(f"🏆 최고 개별 모델: {best_model_name}")
print(f"🎯 최고 RMSE: {best_rmse:.4f} (${best_rmse*100000:,.0f})")
print(f"🎭 앙상블 RMSE: {ensemble_test_rmse:.4f} (${ensemble_test_rmse*100000:,.0f})")
print(f"📈 앙상블 R² Score: {ensemble_test_r2:.4f}")

print(f"\n📊 프로젝트 통계:")
print(f"- 전체 샘플: {len(housing.data):,}개")
print(f"- 훈련 샘플: {len(X_train):,}개")
print(f"- 특성 개수: {len(feature_names)}개 (원본 8개 + 엔지니어링 4개)")
print(f"- 모델 개수: {len(models)}개")

print(f"\n💡 핵심 인사이트:")
print(f"- 중간 소득(MedInc)이 집값 예측에 가장 중요한 특성")
print(f"- 지리적 위치(위도, 경도)도 집값에 큰 영향")
print(f"- 딥러닝으로 R² {ensemble_test_r2:.1%} 달성")
print(f"- 앙상블 기법으로 개별 모델 대비 성능 향상")

print(f"\n📈 모델별 최종 성능 (테스트 데이터):")
sorted_models = sorted(zip(model_names_with_ensemble[:-1], test_rmses[:-1]), 
                      key=lambda x: x[1])
for i, (name, rmse) in enumerate(sorted_models):
    print(f"{i+1}. {name:>10s}: RMSE {rmse:.4f} (${rmse*100000:>7,.0f})")
print(f"   {'Ensemble':>10s}: RMSE {ensemble_test_rmse:.4f} (${ensemble_test_rmse*100000:>7,.0f}) ⭐")

print(f"\n🚀 개선 방향:")
print(f"1. 🔧 추가 특성 엔지니어링")
print(f"   - 다항식 특성 추가")
print(f"   - 지역별 클러스터링 특성")
print(f"   - 교호작용 특성")

print(f"\n2. 🎯 모델 개선")
print(f"   - 베이지안 최적화로 하이퍼파라미터 튜닝")
print(f"   - Gradient Boosting과의 앙상블")
print(f"   - 더 깊은 네트워크 실험")

print(f"\n3. 📊 데이터 개선")
print(f"   - 외부 데이터 (학군, 범죄율 등) 추가")
print(f"   - 시계열 정보 활용")
print(f"   - 공간 자기상관 모델링")

print(f"\n🔗 다음 단계:")
print(f"1. 다른 회귀 문제 도전 (Boston Housing, Bike Sharing 등)")
print(f"2. 실제 부동산 API 데이터로 실시간 예측 시스템 구축")
print(f"3. 설명 가능한 AI 기법 적용 (SHAP, LIME)")
print(f"4. 웹 애플리케이션으로 배포")

print(f"\n📚 학습 포인트:")
print(f"- PyTorch를 사용한 회귀 문제 해결")
print(f"- 특성 엔지니어링의 중요성")
print(f"- 앙상블 기법의 효과")
print(f"- 모델 평가 및 시각화 기법")